# Figure 1: Multiscale Thermal-Hydraulic Dynamics & Flow Boiling Instabilities
### Master Interactive Notebook for High-Impact Publication Plotting

This notebook gives you **complete granular control** over every individual panel and the overall multi-panel composite figure.

---

### **Architecture of this Notebook:**
1. **Cell 1**: Prerequisites, Physics Engine & Analytical Equations (Data generation functions).
2. **Cell 2**: Modular Panel Plotting Functions (`draw_panel_a` through `draw_panel_f`).
3. **Cells 3 to 8**: **Individual Panel Sandbox Cells** — tweak styles, colors, annotations, and render any single panel independently with immediate output.
4. **Cell 9**: **Master Multi-Panel Assembly (2 Columns × 3 Rows)** — customize `figsize`, `hspace`, `wspace`, subpanel margins, and export high-resolution 300 DPI publication images.


In [ ]:
# ==============================================================================
# CELL 1: Imports, Global Plotting Styles & Physics Equations
# ==============================================================================
import os
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle, Circle, FancyBboxPatch
import matplotlib.gridspec as gridspec

# Default Academic Serif Typography
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10.5,
    "axes.labelsize": 11,
    "axes.titlesize": 11.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9.2,
    "axes.linewidth": 1.1,
    "grid.linewidth": 0.5,
    "grid.alpha": 0.35,
    "mathtext.fontset": "cm",
})

# Standardized Color Palette (Deep Navy, Crimson, Teal, Amber Gold, Slate)
PALETTE = {
    "navy": "#1a3c6e",
    "crimson": "#b5451b",
    "teal": "#007a78",
    "gold": "#c88a10",
    "slate": "#4a5568",
    "light_blue": "#dbeafe",
    "light_orange": "#ffedd5",
    "bubble_ec": "#ea580c",
    "bubble_fc": "#fed7aa",
    "callout_bg": "#fff7ed",
    "callout_border": "#ea580c",
    "panel_bg": "#f8fafc",
}

# --- Analytical Physics Calculations ---
def compute_void_fraction(x_arr, pressure_bar=70.0):
    # Homogeneous Equilibrium Model (HEM) Void Fraction alpha(x)
    if pressure_bar >= 50.0:
        rho_f, rho_g = 740.0, 36.0  # ~70 bar (BWR)
    else:
        rho_f, rho_g = 958.0, 0.598  # ~1 bar (Atmospheric)
    gamma = rho_g / rho_f
    alpha = x_arr / (x_arr + (1.0 - x_arr) * gamma)
    return alpha

def compute_scurve_forces(G_arr):
    # Decomposes channel pressure drop into 1-phase friction, 2-phase friction/accel, and gravity
    dp_1phi = 0.22 * G_arr**2 + 0.05 * G_arr
    dp_2phi = 2.4 / (1.0 + 3.0 * G_arr**1.8)
    dp_grav = 0.55 * (1.0 - 0.75 / (1.0 + 2.0 * G_arr**1.5))
    dp_total = dp_1phi + dp_2phi + dp_grav
    return dp_1phi, dp_2phi, dp_grav, dp_total

def compute_ledinegg_roots(G_arr, pump_head=1.35):
    # Calculates static Ledinegg multi-valued equilibrium points
    x = G_arr - 1.6
    dP = x**3 - 1.15 * x + 1.45
    diff = dP - pump_head
    idx_cross = np.where(np.diff(np.sign(diff)) != 0)[0]
    roots = [G_arr[i] for i in idx_cross]
    return dP, roots

def compute_density_wave_time_series(t_arr, tau=2.0):
    # Calculates inlet velocity perturbation and 180-deg delayed exit pressure response
    delta_ui = 0.25 * np.cos(2 * np.pi * t_arr / 4.0)
    delta_dp = 0.35 * np.cos(2 * np.pi * (t_arr - tau) / 4.0)
    return delta_ui, delta_dp

def compute_limit_cycle_phase_orbit():
    # Calculates Hopf bifurcation limit cycle orbit and unstable spiral trajectory
    theta = np.linspace(0, 2*np.pi, 250)
    u_center, lam_center = 0.45, 0.52
    u_orbit = u_center + 0.28 * np.cos(theta) - 0.08 * np.sin(2*theta)
    lam_orbit = lam_center + 0.18 * np.sin(theta) + 0.05 * np.cos(2*theta)
    
    t_spiral = np.linspace(0, 15, 300)
    r_spiral = np.minimum(0.02 * np.exp(0.22 * t_spiral), 1.0)
    u_spiral = u_center + r_spiral * (0.28 * np.cos(t_spiral) - 0.08 * np.sin(2*t_spiral))
    lam_spiral = lam_center + r_spiral * (0.18 * np.sin(t_spiral) + 0.05 * np.cos(2*t_spiral))
    return (u_center, lam_center), (u_orbit, lam_orbit), (u_spiral, lam_spiral)

print("Physics engine & styling prerequisites initialized successfully.")


In [ ]:
# ==============================================================================
# CELL 2: Modular Panel Drawing Functions
# ==============================================================================

def draw_panel_a(ax, title=r"$\mathbf{(a)}$ Physical Zonation & Moving Boundary $\lambda(t)$",
                 z_lambda=2.3, show_bubbles=True, seed=42):
    # Draws Panel (a): Physical Channel Zonation Schematic
    ax.set_xlim(-0.6, 4.6)
    ax.set_ylim(-0.2, 5.8)
    ax.axis("off")
    ax.set_title(title, pad=12, loc="left", fontweight="bold")

    # Plenums
    ax.add_patch(FancyBboxPatch((0.5, -0.1), 3.0, 0.5, boxstyle="round,pad=0.05", 
                                fc="#d0dbe5", ec=PALETTE["navy"], lw=1.5))
    ax.text(2.0, 0.15, r"Lower Plenum ($\Delta P_{\mathrm{ext}}$ Header)", 
            ha="center", va="center", fontsize=9.2, fontweight="bold", color=PALETTE["navy"])

    ax.add_patch(FancyBboxPatch((0.5, 5.0), 3.0, 0.5, boxstyle="round,pad=0.05", 
                                fc="#d0dbe5", ec=PALETTE["navy"], lw=1.5))
    ax.text(2.0, 5.25, r"Upper Plenum (Constant $\Delta P_{\mathrm{ext}}$)", 
            ha="center", va="center", fontsize=9.2, fontweight="bold", color=PALETTE["navy"])

    # Channel walls
    ax.plot([1.0, 1.0], [0.4, 5.0], color=PALETTE["slate"], lw=3.0)
    ax.plot([3.0, 3.0], [0.4, 5.0], color=PALETTE["slate"], lw=3.0)

    # Single phase liquid
    ax.add_patch(Rectangle((1.0, 0.4), 2.0, z_lambda - 0.4, fc=PALETTE["light_blue"], ec="none", alpha=0.85))
    ax.text(2.0, 1.35, "Single-Phase Liquid\n" + r"($0 \leq z < \lambda(t)$)" + "\n" + r"$T < T_{\mathrm{sat}}, \; \rho(z) = \rho_f$", 
            ha="center", va="center", fontsize=8.8, color="#1e40af")

    # Moving saturation boundary
    ax.plot([0.8, 3.2], [z_lambda, z_lambda], color=PALETTE["crimson"], lw=2.2, ls="--")
    ax.text(3.3, z_lambda, r"$\mathbf{z = \lambda(t)}$" + "\n" + r"($h = h_f$ Saturation)", 
            ha="left", va="center", fontsize=8.8, color=PALETTE["crimson"], fontweight="bold")

    # Two-phase mixture
    ax.add_patch(Rectangle((1.0, z_lambda), 2.0, 5.0 - z_lambda, fc=PALETTE["light_orange"], ec="none", alpha=0.85))
    ax.text(2.0, 3.55, "Two-Phase Mixture\n" + r"($\lambda(t) \leq z \leq 1$)" + "\n" + r"$\rho_m(z) \ll \rho_f, \; u(z) \uparrow$", 
            ha="center", va="center", fontsize=8.8, color="#9a3412")

    # Vapor bubbles
    if show_bubbles:
        np.random.seed(seed)
        for _ in range(25):
            bx = np.random.uniform(1.2, 2.8)
            by = np.random.uniform(z_lambda + 0.15, 4.4)
            br = np.random.uniform(0.04, 0.11) * ((by - z_lambda)/2.7 + 0.6)
            ax.add_patch(Circle((bx, by), br, fc=PALETTE["bubble_fc"], ec=PALETTE["bubble_ec"], lw=0.8, alpha=0.9))

    # Heat flux vectors
    for y_pos in np.linspace(0.8, 4.6, 7):
        ax.annotate("", xy=(0.98, y_pos), xytext=(0.4, y_pos),
                    arrowprops=dict(arrowstyle="->", color=PALETTE["crimson"], lw=1.4))
        ax.annotate("", xy=(3.02, y_pos), xytext=(3.6, y_pos),
                    arrowprops=dict(arrowstyle="->", color=PALETTE["crimson"], lw=1.4))
    ax.text(0.12, 2.7, r"Uniform Heat Flux $q''$", ha="center", va="center", rotation=90, 
            fontsize=9.2, color=PALETTE["crimson"], fontweight="bold")

    # Flow velocities
    ax.annotate("", xy=(2.0, 0.75), xytext=(2.0, 0.35), arrowprops=dict(arrowstyle="->", color=PALETTE["navy"], lw=2.0))
    ax.text(1.9, 0.55, r"$u_i(t)$", ha="right", va="center", fontsize=9.5, color=PALETTE["navy"], fontweight="bold")

    ax.annotate("", xy=(2.0, 4.9), xytext=(2.0, 4.3), arrowprops=dict(arrowstyle="->", color="#9a3412", lw=2.5))
    ax.text(1.9, 4.6, r"$u_e(t) \gg u_i$", ha="right", va="center", fontsize=9.5, color="#9a3412", fontweight="bold")


def draw_panel_b(ax, title=r"$\mathbf{(b)}$ Thermodynamic Void Non-Linearity"):
    # Draws Panel (b): 5% Quality Void Fraction Paradox
    ax.set_title(title, pad=12, loc="left", fontweight="bold")
    x = np.linspace(0.0, 1.0, 500)
    alpha_70 = compute_void_fraction(x, pressure_bar=70.0)
    alpha_1 = compute_void_fraction(x, pressure_bar=1.0)

    ax.plot(x * 100, alpha_70 * 100, color=PALETTE["crimson"], lw=2.4, 
            label=r"BWR Core ($70\,\mathrm{bar}, \; \rho_f/\rho_g \approx 20.5$)")
    ax.plot(x * 100, alpha_1 * 100, color=PALETTE["slate"], lw=1.8, ls="--", 
            label=r"Atmospheric ($1\,\mathrm{bar}, \; \rho_f/\rho_g \approx 1600$)")

    # 5% target callout
    x_target = 0.05
    alpha_target = compute_void_fraction(np.array([x_target]), 70.0)[0] * 100

    ax.plot([5.0, 5.0], [0, alpha_target], color=PALETTE["gold"], ls=":", lw=1.6)
    ax.plot([0, 5.0], [alpha_target, alpha_target], color=PALETTE["gold"], ls=":", lw=1.6)
    ax.plot(5.0, alpha_target, "o", color=PALETTE["crimson"], ms=8, zorder=5, mec="black", mew=1.2)

    ax.annotate(r"$\mathbf{5\%}$ Mass Quality ($x=0.05$)" + "\n" + rf"$\Rightarrow \mathbf{{\alpha = {alpha_target:.1f}\%}}$ Channel Volume!",
                xy=(5.0, alpha_target), xytext=(22, 36),
                fontsize=8.8, color="#9a3412", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.4", fc=PALETTE["callout_bg"], ec=PALETTE["callout_border"], lw=1.1),
                arrowprops=dict(arrowstyle="->", color=PALETTE["callout_border"], lw=1.3))

    ax.set_xlabel(r"Thermodynamic Mass Quality, $x \; (\%)$")
    ax.set_ylabel(r"Cross-Sectional Void Fraction, $\alpha \; (\%)$")
    ax.set_xlim(-2, 102)
    ax.set_ylim(-2, 102)
    ax.grid(True)
    ax.legend(loc="lower right", frameon=True, framealpha=0.92, facecolor="white", edgecolor="#e2e8f0")


def draw_panel_c(ax, title=r"$\mathbf{(c)}$ Hydrodynamic Force Decomposition"):
    # Draws Panel (c): S-Curve Force Balance Decomposition
    ax.set_title(title, pad=12, loc="left", fontweight="bold")
    G = np.linspace(0.1, 3.2, 500)
    dp_1phi, dp_2phi, dp_grav, dp_total = compute_scurve_forces(G)

    ax.plot(G, dp_1phi, color=PALETTE["teal"], lw=1.8, ls="-.", label=r"Single-Phase Friction ($\propto G^2$)")
    ax.plot(G, dp_2phi, color=PALETTE["gold"], lw=1.8, ls="--", label=r"Two-Phase Expansion ($\propto 1/\rho_m$)")
    ax.plot(G, dp_grav, color=PALETTE["slate"], lw=1.6, ls=":", label=r"Gravitational Head ($\propto \rho_m g$)")
    ax.plot(G, dp_total, color=PALETTE["navy"], lw=2.6, label=r"$\mathbf{Total \; Internal \; \Delta P}$ (S-Curve)")

    # Negative slope shading
    idx_neg = np.where(np.gradient(dp_total, G) < 0)[0]
    ax.axvspan(G[idx_neg[0]], G[idx_neg[-1]], color="#fee2e2", alpha=0.55, 
               label=r"Negative Slope ($\frac{\partial \Delta P}{\partial G} < 0$)")

    ax.set_xlabel(r"Mass Flow Rate, $G \; (\mathrm{kg/m^2 s})$")
    ax.set_ylabel(r"Channel Pressure Drop, $\Delta P$")
    ax.set_xlim(0.1, 3.2)
    ax.set_ylim(0.0, 3.2)
    ax.grid(True)
    ax.legend(loc="upper right", frameon=True, framealpha=0.92, facecolor="white", edgecolor="#e2e8f0", fontsize=8.4)


def draw_panel_d(ax, title=r"$\mathbf{(d)}$ Static Ledinegg Instability & Flow Excursion"):
    # Draws Panel (d): Multi-Valued Ledinegg Excursion & Runaway Flow Jump
    ax.set_title(title, pad=12, loc="left", fontweight="bold")
    G_d = np.linspace(0.2, 3.0, 500)
    pump_head = 1.35
    dP_d, g_roots = compute_ledinegg_roots(G_d, pump_head=pump_head)

    ax.plot(G_d, dP_d, color=PALETTE["navy"], lw=2.4, label=r"Channel Characteristic $\Delta P_{\mathrm{channel}}(G)$")
    ax.plot(G_d, np.full_like(G_d, pump_head), color=PALETTE["crimson"], lw=2.0, ls="--", label=r"External Pump Supply $\Delta P_{\mathrm{ext}}$")

    colors = [PALETTE["teal"], PALETTE["crimson"], PALETTE["teal"]]
    for gr, col in zip(g_roots, colors):
        ax.plot(gr, pump_head, "o", color=col, ms=8, zorder=5, mec="black", mew=1.2)

    # Point annotations
    ax.annotate(r"$\mathbf{A}$ (Vapor-Rich)", (g_roots[0], pump_head), xytext=(0.7, pump_head + 0.72),
                fontsize=8.6, ha="center", color=PALETTE["teal"], fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=PALETTE["teal"], lw=1.2))

    ax.annotate(r"$\mathbf{B}$ (Unstable Saddle)" + "\n" + r"$\mathbf{\partial \Delta P / \partial G < 0}$", 
                (g_roots[1], pump_head), xytext=(g_roots[1], pump_head - 0.68),
                fontsize=8.6, ha="center", color=PALETTE["crimson"], fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=PALETTE["crimson"], lw=1.2))

    ax.annotate(r"$\mathbf{C}$ (Liquid-Rich)", (g_roots[2], pump_head), xytext=(2.4, pump_head + 0.72),
                fontsize=8.6, ha="center", color=PALETTE["teal"], fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=PALETTE["teal"], lw=1.2))

    # Flow jump trajectory
    ax.annotate("", xy=(g_roots[0] + 0.12, pump_head + 0.15), xytext=(g_roots[1] - 0.12, pump_head + 0.15),
                arrowprops=dict(arrowstyle="->", color=PALETTE["crimson"], lw=2.0, ls=":", connectionstyle="arc3,rad=-0.28"))
    ax.text(1.08, 1.85, "Runaway Flow Collapse\n" + r"$\mathbf{B \rightarrow A}$ (Burnout / CHF)", 
            fontsize=8.4, color=PALETTE["crimson"], fontweight="bold", ha="center")

    ax.set_xlabel(r"Mass Flow Rate, $G$")
    ax.set_ylabel(r"Pressure Drop, $\Delta P$")
    ax.set_xlim(0.2, 3.0)
    ax.set_ylim(0.2, 2.5)
    ax.grid(True)
    ax.legend(loc="lower right", frameon=True, framealpha=0.92, facecolor="white", edgecolor="#e2e8f0", fontsize=8.6)


def draw_panel_e(ax, title=r"$\mathbf{(e)}$ Dynamic Density-Wave Transit Delay $\tau$"):
    # Draws Panel (e): Acoustic 180-deg Phase Shift & Regenerative Delay
    ax.set_title(title, pad=12, loc="left", fontweight="bold")
    t = np.linspace(0, 12, 500)
    delta_ui, delta_dp_exit = compute_density_wave_time_series(t, tau=2.0)

    ax.plot(t, delta_ui, color=PALETTE["navy"], lw=2.2, label=r"Inlet Flow Perturbation $\delta u_i(t)$")
    ax.plot(t, delta_dp_exit, color=PALETTE["crimson"], lw=2.2, ls="--", label=r"Exit Pressure Feedback $\delta \Delta P_{\mathrm{exit}}(t)$")

    t_peak1, t_peak2 = 4.0, 6.0
    ax.plot([t_peak1, t_peak1], [-0.52, 0.42], color=PALETTE["slate"], ls=":", lw=1.3)
    ax.plot([t_peak2, t_peak2], [-0.52, 0.42], color=PALETTE["slate"], ls=":", lw=1.3)

    ax.annotate("", xy=(t_peak2, 0.38), xytext=(t_peak1, 0.38),
                arrowprops=dict(arrowstyle="<->", color=PALETTE["gold"], lw=2.0))
    ax.text((t_peak1 + t_peak2)/2, 0.45, r"$\mathbf{\tau = T/2 \; (180^\circ \; Lag)}$", 
            ha="center", va="bottom", fontsize=9.0, color="#9a3412", fontweight="bold")

    ax.text(6.0, -0.42, r"$\mathbf{Acoustic \; Feedback:}$ Delayed steam packet exits channel" + "\n" + 
            r"$\Rightarrow \Delta P$ pressure spike $\Rightarrow$ Further chokes inlet velocity $u_i$", 
            ha="center", va="center", fontsize=8.4,
            bbox=dict(boxstyle="round,pad=0.35", fc=PALETTE["panel_bg"], ec="#cbd5e1", lw=1.0))

    ax.set_xlabel(r"Dimensionless Time, $t$")
    ax.set_ylabel(r"Perturbation Amplitude")
    ax.set_xlim(0, 12)
    ax.set_ylim(-0.55, 0.65)
    ax.grid(True)
    ax.legend(loc="upper right", frameon=True, framealpha=0.92, facecolor="white", edgecolor="#e2e8f0", fontsize=8.4)


def draw_panel_f(ax, title=r"$\mathbf{(f)}$ Supercritical Hopf Limit-Cycle Orbit"):
    # Draws Panel (f): Limit Cycle Attractor & Inset Time Series
    ax.set_title(title, pad=12, loc="left", fontweight="bold")
    (u_c, lam_c), (u_orb, lam_orb), (u_sp, lam_sp) = compute_limit_cycle_phase_orbit()

    ax.plot(u_sp[:200], lam_sp[:200], color=PALETTE["gold"], lw=1.4, ls=":", label=r"Unstable Spiral ($\mathrm{Re}(\mu) > 0$)")
    ax.plot(u_orb, lam_orb, color=PALETTE["crimson"], lw=2.6, label=r"Stable Limit Cycle $\Gamma$")
    ax.plot(u_c, lam_c, "x", color="black", ms=9, mew=2.2, label=r"Unstable Focus $\mathbf{x}^*$")

    ax.annotate("", xy=(u_orb[50], lam_orb[50]), xytext=(u_orb[45], lam_orb[45]),
                arrowprops=dict(arrowstyle="->", color=PALETTE["crimson"], lw=2.2))
    ax.annotate("", xy=(u_orb[150], lam_orb[150]), xytext=(u_orb[145], lam_orb[145]),
                arrowprops=dict(arrowstyle="->", color=PALETTE["crimson"], lw=2.2))

    # Inset time-series
    inset_ax = ax.inset_axes([0.52, 0.10, 0.44, 0.34])
    t_sim = np.linspace(0, 20, 300)
    u_sim = u_c + 0.28 * np.cos(1.83 * t_sim) * (1.0 - np.exp(-0.35 * t_sim))
    inset_ax.plot(t_sim, u_sim, color=PALETTE["crimson"], lw=1.5)
    inset_ax.set_title(r"Sustained $u_i(t)$ Oscillations", fontsize=7.8, pad=2)
    inset_ax.set_xlabel(r"$t$", fontsize=7.2, labelpad=1)
    inset_ax.set_ylabel(r"$u_i$", fontsize=7.2, labelpad=1)
    inset_ax.tick_params(labelsize=6.8)
    inset_ax.grid(True, alpha=0.3)

    ax.set_xlabel(r"Inlet Liquid Velocity, $u_i(t)$")
    ax.set_ylabel(r"Boiling Boundary Position, $\lambda(t)$")
    ax.set_xlim(0.1, 0.82)
    ax.set_ylim(0.25, 0.8)
    ax.grid(True)
    ax.legend(loc="upper left", frameon=True, framealpha=0.92, facecolor="white", edgecolor="#e2e8f0", fontsize=8.4)

print("Modular plotting functions registered.")


In [ ]:
# ==============================================================================
# CELL 3: [SANDBOX] Panel (a) — Physical Zonation & Moving Boundary
# ==============================================================================
fig, ax = plt.subplots(figsize=(7.5, 6.0), dpi=150)

# Custom Styling Options for Panel (a):
CUSTOM_TITLE_A = r"$\mathbf{(a)}$ Physical Zonation & Moving Boundary $\lambda(t)$"
Z_LAMBDA_VAL = 2.3
SHOW_BUBBLES = True
RANDOM_SEED = 42

draw_panel_a(ax, title=CUSTOM_TITLE_A, z_lambda=Z_LAMBDA_VAL, show_bubbles=SHOW_BUBBLES, seed=RANDOM_SEED)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 4: [SANDBOX] Panel (b) — Void Non-Linearity & Quality Paradox
# ==============================================================================
fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=150)

# Custom Styling Options for Panel (b):
CUSTOM_TITLE_B = r"$\mathbf{(b)}$ Thermodynamic Void Non-Linearity"

draw_panel_b(ax, title=CUSTOM_TITLE_B)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 5: [SANDBOX] Panel (c) — Hydrodynamic Force Decomposition
# ==============================================================================
fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=150)

# Custom Styling Options for Panel (c):
CUSTOM_TITLE_C = r"$\mathbf{(c)}$ Hydrodynamic Force Decomposition"

draw_panel_c(ax, title=CUSTOM_TITLE_C)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 6: [SANDBOX] Panel (d) — Static Ledinegg Instability & Excursion
# ==============================================================================
fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=150)

# Custom Styling Options for Panel (d):
CUSTOM_TITLE_D = r"$\mathbf{(d)}$ Static Ledinegg Instability & Flow Excursion"

draw_panel_d(ax, title=CUSTOM_TITLE_D)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 7: [SANDBOX] Panel (e) — Dynamic Density-Wave Transit Delay
# ==============================================================================
fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=150)

# Custom Styling Options for Panel (e):
CUSTOM_TITLE_E = r"$\mathbf{(e)}$ Dynamic Density-Wave Transit Delay $\tau$"

draw_panel_e(ax, title=CUSTOM_TITLE_E)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 8: [SANDBOX] Panel (f) — Supercritical Hopf Limit-Cycle Orbit
# ==============================================================================
fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=150)

# Custom Styling Options for Panel (f):
CUSTOM_TITLE_F = r"$\mathbf{(f)}$ Supercritical Hopf Limit-Cycle Orbit"

draw_panel_f(ax, title=CUSTOM_TITLE_F)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 9: [MASTER ASSEMBLY] 2-Column × 3-Row Publication Figure (High-Res 300 DPI)
# ==============================================================================
# Global Multi-Panel Dimensions & Spacing Controls (Adjust here for perfect margins!)
FIG_WIDTH = 15.5       # Total width in inches
FIG_HEIGHT = 17.0      # Total height in inches (2 cols x 3 rows layout)
HSPACE = 0.30          # Vertical spacing between rows
WSPACE = 0.25          # Horizontal spacing between columns
OUTPUT_DPI = 300       # Publication standard DPI

fig = plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT), dpi=OUTPUT_DPI)
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=HSPACE, wspace=WSPACE)

# Row 1: Physical Zonation (Left) & Thermodynamic Void Non-Linearity (Right)
ax1 = fig.add_subplot(gs[0, 0])
draw_panel_a(ax1)

ax2 = fig.add_subplot(gs[0, 1])
draw_panel_b(ax2)

# Row 2: Hydrodynamic Force Decomposition (Left) & Static Ledinegg Jump (Right)
ax3 = fig.add_subplot(gs[1, 0])
draw_panel_c(ax3)

ax4 = fig.add_subplot(gs[1, 1])
draw_panel_d(ax4)

# Row 3: Dynamic Density-Wave Transit Delay (Left) & Supercritical Limit-Cycle Orbit (Right)
ax5 = fig.add_subplot(gs[2, 0])
draw_panel_e(ax5)

ax6 = fig.add_subplot(gs[2, 1])
draw_panel_f(ax6)

# Output Paths
os.makedirs("d:/AGravity/Tide_Tutor/manuscript/figures", exist_ok=True)
save_path_png = "d:/AGravity/Tide_Tutor/manuscript/figures/fig1_multiscale_dynamics_2x3.png"
save_path_pdf = "d:/AGravity/Tide_Tutor/manuscript/figures/fig1_multiscale_dynamics_2x3.pdf"

fig.savefig(save_path_png, dpi=OUTPUT_DPI, bbox_inches="tight")
fig.savefig(save_path_pdf, bbox_inches="tight")
print(f"Master 2x3 Figure saved successfully to:\n  - {save_path_png}\n  - {save_path_pdf}")

plt.show()
